## Explore all Fluree v4 branch management endpoints
### Create a brabch for a given ledger
```python
POST /branch
POST http://localhost:8090/v1/fluree/branch
```



In [1]:
import requests
url = "http://localhost:8090/v1/fluree/branch"
payload= {
    "ledger":"demo",
    "branch":"feature-test",
    "source": "dev"
}

response = requests.post(url, json=payload)
print(response.json())

{'error': 'Internal error: Source branch demo:dev has no commit head', 'status': 500, '@type': 'err:system/InternalError'}


**Note**: It is normal to face the error:
```json

{'error': 'Internal error: Source branch demo:dev has no commit head', 'status': 500, '@type': 'err:system/InternalError'}
```
- the source branch that is demo:dev has no latest commit pointing to any data. The branch demo:dev is there, but no commit was made.
- The commit history may have been removed accidentally.
- The application is looking for demo:dev, but the actual branch may be main, master, or another name.
- In distributed databases (such as Fluree), metadata references a branch that has not been fully synchronized.

### For a given ledger, list all the branches: 
```python
GET /branch/{ledger-name}
```


In [2]:
url = f"http://localhost:8090/v1/fluree/branch/demo"
response = requests.get(url)
print(response.json())
print(response.status_code)

[{'branch': 'dev', 'ledger_id': 'demo:dev', 't': 0}]
200


- branch: means branch name
- ledger_id: Full ledger:branch identifier
- t: Current transaction time on this branch
- source: Source branch (only present for branches created via /branch)

### Drop a branch
```python
POST /drop-branch
```
Drop a branch from a ledger. Admin-protected


In [4]:
url = f"http://localhost:8090/v1/fluree/drop-branch"
payload = {
    "ledger":"demo",
    "branch":"dev"
}

response = requests.post(url, json=payload)
print(response.json())
print(response.status_code)

{'error': "Cannot drop 'dev': it is the root of ledger 'demo'. Use drop_ledger to remove the whole ledger.", 'status': 400, '@type': 'err:system/InternalError'}
400


- ledger_id:  Full ledger:branch identifier of the dropped branch
- status: Drop status
- deferred: True if the branch has children: retracted but storage preserved
- filed_deleted: Number of storage artifacts removed; omitted when zero
- cascade: List of ancestors branch ledger_ids that were cascade-dropped; omitted when empty
- warnings: Any non-fatal warnings during the drop; omitted when empty

**Behavior**:

- Cannot drop main: Returns 400 bad request
- Leaf branch (no children): Fully drops, deletes storage artifacts, purges, Nsrecord, decrements parent's child count. If the parent was previously retracted and its child count reaches 0, the parent is cascade-dropped too
- Branch with children (branches > 0): Retracted (hidden from listings, rejects new transactions) but storage is preserved for children. When the last child is eventually dropped, the retracted parent is cascade-purged automatically.
